In [1]:
import fine as fn
import pyomo.environ as pyomo

# Step 1: Define the Energy System Model
esM = fn.EnergySystemModel(
    locations={"A"},
    onlycommodities={"electricity", "hydrogen"},
    onlycommodityUnitsDict={"electricity": "GW", "hydrogen": "kg"},
    onlymaterials={"steel", "copper", "iron"},
    onlymaterialUnitsDict={"steel": "tons", "copper": "kg", "iron": "kg"}
)
esM.pyM = pyomo.ConcreteModel()

In [2]:
# Check if commodity declarations work
print("only Materials Units Dict:", esM.onlymaterialUnitsDict)
print("only Commodity Units Dict:", esM.onlycommodityUnitsDict)
print("Commodities:", esM.commodities)
print("Commodity Units Dict:", esM.commodityUnitsDict)


only Materials Units Dict: {'steel': 'tons', 'copper': 'kg', 'iron': 'kg'}
only Commodity Units Dict: {'electricity': 'GW', 'hydrogen': 'kg'}
Commodities: ['electricity', 'hydrogen', 'copper', 'iron', 'steel']
Commodity Units Dict: {'electricity': 'GW', 'hydrogen': 'kg', 'copper': 'kg', 'iron': 'kg', 'steel': 'tons'}


In [3]:
# Step 2: Add a Energy Source Component that Requires Materials                             
esM.add(
    fn.Source(
        esM=esM, 
        name="Wind Turbines",
        commodity="electricity",
        hasCapacityVariable=True,
        materialIntensity={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
)

# Add a Energy Storage Component that Requires Materials
esM.add(
    fn.Storage(
        esM=esM,
        name="Battery",
        commodity="electricity",
        chargeEfficiency=0.9,
        dischargeEfficiency=0.9,
        materialIntensity={"steel": 2, "copper": 0.5},  # Materials required for commissioning
        materialRecovery={"steel": 0.8, "copper": 0.3}   # Recovery fractions at decommissioning
    )
) 

# Add a sink 

In [4]:
# Step 3: Add Material Source 
esM.add(
    fn.Source(
        esM=esM, 
        name="Steel Supply",
        material="steel",
        #materialConsumption="steel",
        #commodity="steel",
        hasCapacityVariable=True,
    )
)

source = esM.add(
    fn.Source(
        esM=esM, 
        name="Copper Supply",
        material="copper",
        #materialConsumption="steel",
        #commodity="copper",
        hasCapacityVariable=True,
    )
)
#esM.source.commodity

In [5]:
# Step 4: Add Energy Sink Component that consumes Energy 
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Electricity demand",
        commodity="electricity",
        hasCapacityVariable=False,
        operationRateFix=50,
    )
)

In [6]:
# Step 5: Add Material Sink that consumes Materials 
sink = esM.add(
    fn.Sink(
        esM=esM,
        name="Steel demand",
        hasCapacityVariable=False,
        material="steel",
    )
)

In [7]:
# Create a Sink with materials=True and commodity="steel"
#sink = fn.Sink(
#    esM=esM,
#    name="Steel demand",
#    commodity="steel",
#    hasCapacityVariable=False,
#    materials=True,
#)

#sink.onlycommodityUnit

source = fn.Source(
    esM=esM, 
    name="Copper Supply",
    material="copper",
    #materialConsumption="steel",
    #commodity="copper",
    hasCapacityVariable=True,
)

source.onlymaterials

'copper'

In [8]:
esM.optimize()

Declaring sets, variables and constraints for SourceSinkModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(0.5976 sec)

Declaring sets, variables and constraints for StorageModel
	declaring sets... 
	declaring variables... 
	declaring constraints... 
		(1.4612 sec)

Declaring shared potential constraint...
		(0.0004 sec)

Declaring linked component quantity constraint...
		(0.0000 sec)

Declaring commodity balances...
ERROR: Constructing component 'locationCommoditySet' from data=None failed:
        AttributeError: 'Source' object has no attribute 'commodity'


AttributeError: 'Source' object has no attribute 'commodity'

In [ ]:
esM.getOptimizationSummary("SourceSinkModel", outputLevel=2)

A
Component          Property      Unit              
Electricity        capacity      [GW]          50.0
                   commissioning [GW]          50.0
                   operation     [GW*h/a]  438000.0
                                 [GW*h]    438000.0
Electricity demand operation     [GW*h/a]  438000.0
                                 [GW*h]    438000.0

In [ ]:
import pyomo.environ as pyomo

# Testmodell
class EnergySystemModel:
    def __init__(self):
        self.investmentPeriods = [0, 1, 2]  # Beispiel: Investitionsperioden
        # Beispiel für ein Dictionary, das Komponenten darstellt
        self.componentsDict = {
            'comp1': {
                'processedLocationalEligibility': {
                    'loc1': 1, 'loc2': 1
                },
                'materialIntensity': {'mat1': 2, 'mat2': 3},
                'materialRecovery': {'mat3': 1},
            }
        }

# Pyomo Modell-Setup
def declareMaterialVarSet(self, pyM, esM):
    compDict, abbrvName = self.componentsDict, self.abbrvName

    def declareMaterialSet(pyM):
        return (
            (loc, compName, mat, ip)
            for compName, comp in compDict.items()
            for loc in comp['processedLocationalEligibility'].keys()
            for mat in comp['materialIntensity'].keys() | comp['materialRecovery'].keys() 
            for ip in esM.investmentPeriods
        )

    setattr(
        pyM,
        "materialSet_" + abbrvName,
        pyomo.Set(dimen=4, initialize=declareMaterialSet),
    )

# Testfunktion
def test_declareMaterialVarSet():
    # Erstelle ein Beispiel-Modell und EnergySystemModel
    pyM = pyomo.ConcreteModel()
    esM = EnergySystemModel()
    esM.componentsDict = {
        'comp1': {
            'processedLocationalEligibility': {'loc1': 1, 'loc2': 1},
            'materialIntensity': {'mat1': 2, 'mat2': 3},
            'materialRecovery': {'mat3': 1},
        }
    }
    esM.investmentPeriods = [0, 1, 2]  # Beispiel: Investitionsperioden
    abbrvName = "comp1"  # Abkürzung für die Komponente
    
    # Setze die Attribute des Tests
    esM.abbrvName = abbrvName
    esM.pyM = pyM
    esM.componentsDict = esM.componentsDict  # Komponenten einfügen
    
    # Aufruf der declareMaterialVarSet Funktion
    declareMaterialVarSet(esM, pyM, esM)

    # Teste, ob das Set erfolgreich im Pyomo Modell definiert wurde
    materialSet = getattr(pyM, "materialSet_" + abbrvName)
    
    print("materialSet_:", materialSet)
    
    # Teste, ob der Inhalt des Sets korrekt ist
    for item in materialSet:
        print(f"Set Element: {item}")

# Testfunktion ausführen
test_declareMaterialVarSet()


materialSet_: materialSet_comp1
Set Element: ('loc1', 'comp1', 'mat2', 0)
Set Element: ('loc1', 'comp1', 'mat2', 1)
Set Element: ('loc1', 'comp1', 'mat2', 2)
Set Element: ('loc1', 'comp1', 'mat1', 0)
Set Element: ('loc1', 'comp1', 'mat1', 1)
Set Element: ('loc1', 'comp1', 'mat1', 2)
Set Element: ('loc1', 'comp1', 'mat3', 0)
Set Element: ('loc1', 'comp1', 'mat3', 1)
Set Element: ('loc1', 'comp1', 'mat3', 2)
Set Element: ('loc2', 'comp1', 'mat2', 0)
Set Element: ('loc2', 'comp1', 'mat2', 1)
Set Element: ('loc2', 'comp1', 'mat2', 2)
Set Element: ('loc2', 'comp1', 'mat1', 0)
Set Element: ('loc2', 'comp1', 'mat1', 1)
Set Element: ('loc2', 'comp1', 'mat1', 2)
Set Element: ('loc2', 'comp1', 'mat3', 0)
Set Element: ('loc2', 'comp1', 'mat3', 1)
Set Element: ('loc2', 'comp1', 'mat3', 2)


In [ ]:
import pyomo.environ as pyomo

# Testmodell
class EnergySystemModel:
    def __init__(self):
        self.investmentPeriods = [0, 1, 2]  # Beispiel: Investitionsperioden
        self.componentsDict = {
            'comp1': {
                'processedLocationalEligibility': {'loc1': 1, 'loc2': 1},
                'materialIntensity': {'mat1': 2, 'mat2': 3},
                'materialRecovery': {'mat3': 1},
            }
        }

# Funktion zum Setzen des Material-Sets
def declareMaterialVarSet(self, pyM, esM):
    compDict, abbrvName = self.componentsDict, self.abbrvName

    def declareMaterialSet(pyM):
        return (
            (loc, mat, ip, compName)
            for compName, comp in compDict.items()
            for loc in comp['processedLocationalEligibility'].keys()
            for mat in comp['materialIntensity'].keys() | comp['materialRecovery'].keys() 
            for ip in esM.investmentPeriods
        )

    setattr(
        pyM,
        "materialSet_" + abbrvName,
        pyomo.Set(dimen=4, initialize=declareMaterialSet),
    )

# Funktion zum Setzen der Materialverbrauchs- und Rückgewinnungsvariablen
def declareMaterialVars(self, pyM, esM):
    abbrvName = self.abbrvName

    setattr(
        pyM,
        "materialIntensity_" + abbrvName,
        pyomo.Var(
            getattr(pyM, "materialSet_" + abbrvName),
            domain=pyomo.NonNegativeReals,
        ),
    )

    setattr(
        pyM,
        "materialRecovery_" + abbrvName,
        pyomo.Var(
            getattr(pyM, "materialSet_" + abbrvName),
            domain=pyomo.NonNegativeReals,
        ),
    )

# **Testfunktion**
def test_declareMaterialVars():
    # Erstelle das Pyomo-Modell und das EnergySystemModel
    pyM = pyomo.ConcreteModel()
    esM = EnergySystemModel()
    abbrvName = "comp1"

    # Setze die Attribute für den Test
    esM.abbrvName = abbrvName
    esM.pyM = pyM

    # 1️⃣ Set deklarieren
    declareMaterialVarSet(esM, pyM, esM)

    # 2️⃣ Variablen deklarieren
    declareMaterialVars(esM, pyM, esM)

    # 3️⃣ Überprüfe, ob das Set erstellt wurde
    materialSet = getattr(pyM, "materialSet_" + abbrvName)
    print("\n✅ materialSet erfolgreich erstellt!")
    for item in materialSet:
        print(f"Set Element: {item}")

    # 4️⃣ Überprüfe, ob die Variablen existieren
    materialIntensity_ = getattr(pyM, "materialIntensity_" + abbrvName)
    materialRecovery = getattr(pyM, "materialRecovery_" + abbrvName)

    print("\n✅ Materialverbrauchs- und Rückgewinnungsvariablen erstellt!")

    # 5️⃣ Teste Beispielwerte zuweisen
    for item in materialSet:
        materialIntensity_[item] = 5  # Setzt beispielhaft 5 für alle Einträge
        materialRecovery[item] = 2  # Setzt beispielhaft 2 für alle Einträge

    # 6️⃣ Ausgabe der Variablenwerte
    print("\n📌 Beispielwerte für Material Intensity:")
    for item in materialSet:
        print(f"{item}: {materialIntensity_[item].value}")

    print("\n📌 Beispielwerte für Material Recovery:")
    for item in materialSet:
        print(f"{item}: {materialRecovery[item].value}")

# **Testfunktion ausführen**
test_declareMaterialVars()



✅ materialSet erfolgreich erstellt!
Set Element: ('loc1', 'mat2', 0, 'comp1')
Set Element: ('loc1', 'mat2', 1, 'comp1')
Set Element: ('loc1', 'mat2', 2, 'comp1')
Set Element: ('loc1', 'mat1', 0, 'comp1')
Set Element: ('loc1', 'mat1', 1, 'comp1')
Set Element: ('loc1', 'mat1', 2, 'comp1')
Set Element: ('loc1', 'mat3', 0, 'comp1')
Set Element: ('loc1', 'mat3', 1, 'comp1')
Set Element: ('loc1', 'mat3', 2, 'comp1')
Set Element: ('loc2', 'mat2', 0, 'comp1')
Set Element: ('loc2', 'mat2', 1, 'comp1')
Set Element: ('loc2', 'mat2', 2, 'comp1')
Set Element: ('loc2', 'mat1', 0, 'comp1')
Set Element: ('loc2', 'mat1', 1, 'comp1')
Set Element: ('loc2', 'mat1', 2, 'comp1')
Set Element: ('loc2', 'mat3', 0, 'comp1')
Set Element: ('loc2', 'mat3', 1, 'comp1')
Set Element: ('loc2', 'mat3', 2, 'comp1')

✅ Materialverbrauchs- und Rückgewinnungsvariablen erstellt!

📌 Beispielwerte für Material Intensity:
('loc1', 'mat2', 0, 'comp1'): 5
('loc1', 'mat2', 1, 'comp1'): 5
('loc1', 'mat2', 2, 'comp1'): 5
('loc1', 